# Hotel booking analysis
Executed walkthrough of the 25 questions. Run from the extracted hotel project folder. The source covers scheduled arrivals from July 2015 through August 2017.

In [1]:
from pathlib import Path
import runpy
state = runpy.run_path(str(Path('analyze.py').resolve()))
d = state['d']
tables = state['tables']
summary = state['summary']

{
  "bookings": 119390,
  "canceled": 44224,
  "cancellation_rate": 0.37041628277075134,
  "mean_lead": 104.01141636652986,
  "median_lead": 69.0,
  "mean_requests": 0.5713627607002262,
  "parking_bookings": 7416,
  "parking_rate": 0.06211575508836586,
  "mean_week_nights": 2.500301532791691,
  "mean_weekend_nights": 0.9275986263506156,
  "mean_total_nights": 3.4279001591423066,
  "agent_bookings": 103050,
  "agent_rate": 0.8631376162157635,
  "mean_adr": 101.83112153446687,
  "completed_value_proxy": 25996324.21,
  "completed_proxy_nights": 255030,
  "completed_proxy_weighted_adr": 101.93437717131319,
  "status_counts": {
    "Check-Out": 75166,
    "Canceled": 43017,
    "No-Show": 1207
  }
}
{
  "source_rows": 119390,
  "source_columns": 32,
  "identical_later_rows": 31994,
  "distinct_rows": 87396,
  "missing": {
    "children": 4,
    "country": 488,
    "agent": 16340,
    "company": 112593
  },
  "zero_guests": 180,
  "zero_nights": 715,
  "negative_adr": 1,
  "adr_above_1000": 

## B01 Average booking lead time
The average lead time is 104.01 days, while the median is 69 days. The higher mean shows that some bookings made far in advance pull the average upward. These are days from booking entry to scheduled arrival, including bookings that later canceled.

Calculation: Average lead_time across all 119,390 records.

In [2]:
print(d.lead_time.describe())

count    119390.000000
mean        104.011416
std         106.863097
min           0.000000
25%          18.000000
50%          69.000000
75%         160.000000
max         737.000000
Name: lead_time, dtype: float64


## B02 Bookings by hotel type
City Hotel has 79,330 records, or 66.45% of the sample. Resort Hotel has 40,060, or 33.55%. This compares two properties with different capacity and business mixes. It does not establish that city hotels are more popular across the industry.

Calculation: Count records by hotel.

In [3]:
print(tables['hotel_summary'].to_string(index=False))

       hotel  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests  MeanPreviousCancellations
  City Hotel     79330     33102          0.417270    109.735724 105.304465       99.9    2.978142      0.546918                   0.079743
Resort Hotel     40060     11122          0.277634     92.675686  94.952930       75.0    4.318547      0.619770                   0.101722


![Bookings by hotel type](charts/hotel_bookings.png)

## B03 Canceled bookings
The is_canceled flag identifies 44,224 records, giving a cancellation rate of 37.04%. The final status splits these into 43,017 Canceled and 1,207 No-Show records. The project uses the supplied flag consistently, so no-shows are included in cancellation totals.

Calculation: Sum is_canceled; divide by the record count.

In [4]:
print(d.is_canceled.sum(), d.is_canceled.mean())
print(d.reservation_status.value_counts())

44224 0.37041628277075134
reservation_status
Check-Out    75166
Canceled     43017
No-Show       1207
Name: count, dtype: int64


## B04 Most common arrival month
August has the most scheduled arrivals in the pooled sample, with 13,877 bookings. July and August each occur in three years; the other months occur in two. After dividing by the number of observed year-months, May leads with 5,895.5 records per observed May. Use the pooled answer for the assignment, and the coverage-adjusted view for planning.

Calculation: Count arrival_date_month; also divide counts by observed year-months.

In [5]:
print(tables['month_of_year'].to_string(index=False))

 MonthNumber     Month  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests  ObservedYearMonths  BookingsPerObservedMonth
           1   January      5929      1807          0.304773     45.310002  70.361241      71.00    2.971496      0.547478                   2               2964.500000
           2  February      8068      2696          0.334160     47.859569  73.582276      75.00    3.031111      0.525657                   2               4034.000000
           3     March      9794      3149          0.321523     70.119257  80.679646      79.20    3.419338      0.497447                   2               4897.000000
           4     April     11089      4524          0.407972     94.822888 100.380790      96.30    3.366850      0.521508                   2               5544.500000
           5       May     11791      4677          0.396658    117.646171 108.695516     108.00    3.276567      0.532610                   2             

![Most common arrival month](charts/arrival_months.png)

## B05 Special requests per booking
Bookings average 0.571 special requests. Most records have none: 70,318 of 119,390, or 58.90%. Requests can help estimate service workload, but their count does not measure whether the guest was satisfied or whether the request was fulfilled.

Calculation: Average total_of_special_requests.

In [6]:
print(d.total_of_special_requests.mean())
print(tables['requests'].to_string(index=False))

0.5713627607002262
 total_of_special_requests  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
                         0     70318     33556          0.477204    114.284920  94.961611     89.000    3.268651           0.0
                         1     33226      7318          0.220249     88.942425 108.455824    101.875    3.588455           1.0
                         2     12969      2866          0.220989     88.942170 117.033451    110.700    3.747089           2.0
                         3      2497       446          0.178614     92.569083 123.846131    117.850    3.977573           3.0
                         4       340        36          0.105882    110.520588 130.568706    119.425    4.317647           4.0
                         5        40         2          0.050000    105.550000 127.746500    110.440    4.650000           5.0


## B06 Country with the most bookings
PRT, the code for Portugal, has the most records: 48,590, or 40.70% of all bookings. Country is unknown for 488 records and stays separate from known countries. This is a booking-level count, not a count of unique guests.

Calculation: Count country labels; retain missing country as Unknown.

In [7]:
print(tables['countries'].head(10).to_string(index=False))

country  Bookings
    PRT     48590
    GBR     12129
    FRA     10415
    ESP      8568
    DEU      7287
    ITA      3766
    IRL      3375
    BEL      2342
    BRA      2224
    NLD      2104


## B07 ADR by hotel type
City Hotel has mean ADR of 105.30, compared with 94.95 for Resort Hotel. Their medians are 99.90 and 75.00. These booking-weighted means include canceled bookings and the source outliers. ADR is expressed in the source currency units; no exchange-rate conversion is made.

Calculation: Group by hotel and average adr.

In [8]:
print(tables['hotel_summary'].to_string(index=False))

       hotel  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests  MeanPreviousCancellations
  City Hotel     79330     33102          0.417270    109.735724 105.304465       99.9    2.978142      0.546918                   0.079743
Resort Hotel     40060     11122          0.277634     92.675686  94.952930       75.0    4.318547      0.619770                   0.101722


## B08 Bookings requiring parking
7,416 bookings require at least one parking space, representing 6.21% of all records. This percentage uses bookings as the denominator. It cannot be described as a percentage of individual guests, because one booking can contain several guests.

Calculation: Count required_car_parking_spaces > 0; divide by all records.

In [9]:
print(d.HasParking.sum(), d.HasParking.mean())

7416 0.06211575508836586


## B09 Average week and weekend nights
The mean is 2.500 week nights and 0.928 weekend nights per booking, giving 3.428 total nights. The main figures include booked nights for canceled records. They describe stay length, not occupancy rate. Actual occupancy needs occupied room-nights and available room-nights for the same dates.

Calculation: Average stays_in_week_nights and stays_in_weekend_nights; add them for total nights.

In [10]:
print(d[['stays_in_week_nights','stays_in_weekend_nights','TotalNights']].mean())

stays_in_week_nights       2.500302
stays_in_weekend_nights    0.927599
TotalNights                3.427900
dtype: float64


## B10 Bookings through travel agents
103,050 bookings have a travel-agent ID, or 86.31% of the sample. The other 16,340 have no applicable agent. The source paper explains that a NULL agent means the booking did not come through an agent, so these IDs are not filled with a typical agency number.

Calculation: Count non-missing agent IDs; keep IDs categorical.

In [11]:
print(d.HasAgent.sum(), d.HasAgent.mean())

103050 0.8631376162157635


## M01 Cancellation rates by hotel
City Hotel has a cancellation rate of 41.73%: 33,102 out of 79,330 records. Resort Hotel is at 27.76%: 11,122 out of 40,060. The difference is 13.96 percentage points. Channel mix, advance booking and deposit arrangements may contribute to this gap.

Calculation: Calculate sum(is_canceled) / record count within each hotel.

In [12]:
print(tables['hotel_summary'].to_string(index=False))

       hotel  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests  MeanPreviousCancellations
  City Hotel     79330     33102          0.417270    109.735724 105.304465       99.9    2.978142      0.546918                   0.079743
Resort Hotel     40060     11122          0.277634     92.675686  94.952930       75.0    4.318547      0.619770                   0.101722


## M02 ADR by market segment
Online TA has the highest mean ADR at 117.20, followed by Direct at 115.45. Complementary is lowest at 2.89, which fits a segment containing many free or heavily discounted stays. Undefined has only two records and is not useful for a broad comparison. ADR alone does not measure segment profitability because commissions and costs are absent.

Calculation: Group by market_segment; report count, mean and median ADR.

In [13]:
print(tables['market_segments'].to_string(index=False))

market_segment  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
      Aviation       237        52          0.219409      4.443038 100.142110       95.0    3.607595      0.113924
 Complementary       743        97          0.130552     13.286676   2.886366        0.0    1.647376      0.932705
     Corporate      5295       992          0.187347     22.125590  69.358952       65.0    2.092918      0.229839
        Direct     12606      1934          0.153419     49.859115 115.445175      105.0    3.205775      0.576075
        Groups     19811     12097          0.610620    186.973096  79.479472       70.0    2.992529      0.087174
 Offline TA/TO     24219      8311          0.343160    135.004459  87.354783       85.5    3.903877      0.268013
     Online TA     56477     20739          0.367211     82.998725 117.197063      109.8    3.573986      0.899393
     Undefined         2         2          1.000000      1.500000  15.000000   

## M03 Lead time and cancellation
Longer lead times are associated with more cancellations. The rate rises from 9.63% for bookings made 0 to 7 days ahead to 67.66% for those made at least 366 days ahead. Pearson correlation is 0.293 and Spearman is 0.317. The relationship can also reflect group bookings and deposit practices; it is not a causal effect of waiting longer.

Calculation: Correlate lead_time with is_canceled; compare explicit day bands.

In [14]:
print(tables['correlations'].iloc[[0]].to_string(index=False))
print(tables['lead_bands'].to_string(index=False))

  Feature     Outcome      N  Pearson  Spearman
lead_time is_canceled 119390 0.293123  0.316635
   LeadBand  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
     0 to 7     19746      1902          0.096323      2.315304  89.970778      82.88    1.999848      0.543401
    8 to 30     18960      5283          0.278639     17.942194 108.121611      98.00    3.091034      0.680538
   31 to 90     29553     11141          0.376984     57.305857 106.572426      96.30    3.566305      0.618922
  91 to 180     26439     11821          0.447105    132.113091 109.125540     100.00    4.090851      0.579523
 181 to 365     21544     11947          0.554540    253.386326  95.073570      90.00    4.151550      0.474889
366 or more      3148      2130          0.676620    440.459975  78.811569      67.00    2.594663      0.234435


![Lead time and cancellation](charts/lead_cancellations.png)

## M04 Largest distribution channel
TA/TO has 97,870 booking records, or 81.98% of the sample. Direct has 14,645 and Corporate has 6,677. A large channel count does not by itself establish effectiveness, because acquisition cost and net income are not supplied. TA/TO means travel agents and tour operators.

Calculation: Count records by distribution_channel.

In [15]:
print(tables['channels'].to_string(index=False))

distribution_channel  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
           Corporate      6677      1474          0.220758     44.910289  69.325204       65.0    2.380710      0.205781
              Direct     14645      2557          0.174599     57.829362 106.654983       96.0    3.194333      0.534175
                 GDS       193        37          0.191710     20.715026 120.554301      115.0    1.958549      0.196891
               TA/TO     97870     40152          0.410259    115.122438 103.292866       95.5    3.537192      0.602565
           Undefined         5         4          0.800000     23.000000  46.240000       18.0    3.400000      1.400000


## M05 Previous cancellations by hotel
Mean previous cancellations are 0.0797 for City Hotel and 0.1017 for Resort Hotel. These are historical counts recorded on each booking, rather than the current booking’s outcome. Multiple bookings can carry the same guest history, and no guest ID is available for a unique-customer average.

Calculation: Average previous_cancellations by hotel.

In [16]:
print(tables['hotel_summary'][['hotel','MeanPreviousCancellations']].to_string(index=False))

       hotel  MeanPreviousCancellations
  City Hotel                   0.079743
Resort Hotel                   0.101722


## M06 ADR over the years
Mean ADR rises from 87.18 in 2015 to 98.33 in 2016 and 114.64 in 2017. The years cover different months: July to December 2015, all of 2016, and January to August 2017. Restricting the comparison to July and August gives 102.55, 134.63 and 153.24 respectively. Both views rise, but composition changes prevent interpreting the difference as a pure price increase.

Calculation: Average ADR by arrival year; repeat for July and August, present in all years.

In [17]:
print(tables['year_adr'].to_string(index=False))
print(tables['year_adr_jul_aug'].to_string(index=False))

 arrival_date_year  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
              2015     21996      8142          0.370158     97.245999  87.178515      77.76    3.390980      0.435443
              2016     56707     20337          0.358633    102.870369  98.325863      90.95    3.336784      0.548645
              2017     40687     15745          0.386979    109.259223 114.637950     107.10    3.574852      0.676506
 arrival_date_year  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
              2015      6665      2857          0.428657    110.461215 102.553964      94.50    3.771943      0.512228
              2016      9635      3324          0.344992    122.405293 134.632649     124.46    3.929632      0.673897
              2017     10238      3800          0.371166    145.673862 153.241150     140.00    3.983200      0.861594


![ADR over the years](charts/adr_monthly.png)

## M07 Month with the largest lodging value
August leads with an estimated completed-stay lodging value of 4,917,159.06 source currency units. Its average per observed August is 1,639,053.02 and also leads after the coverage adjustment. The assignment’s sum-of-ADR calculation gives August 1,944,327.60, but adding daily rates is not a revenue calculation. The full table retains this requested arithmetic alongside the lodging-value estimate.

Calculation: For Check-Out records with positive nights and nonnegative ADR, sum adr × total nights; allocate the whole stay to arrival month.

In [18]:
print(tables['revenue_month'].to_string(index=False))

 MonthNumber     Month     SumADR  BookedValueProxy  CompletedValueProxy  CompletedProxyNights  EligibleBookings  ObservedYearMonths  ProxyPerObservedMonth
           1   January  417171.80        1273077.43            776676.84                 11332              4072                   2           3.883384e+05
           2  February  593661.80        1818972.70           1147541.48                 15788              5319                   2           5.737707e+05
           3     March  790176.45        2639284.89           1652162.39                 21801              6580                   2           8.260812e+05
           4     April 1113122.58        3710466.14           2099857.48                 21795              6535                   2           1.049929e+06
           5       May 1281628.83        4108125.90           2395213.64                 23616              7037                   2           1.197607e+06
           6      June 1276277.11        4499387.28           25

![Month with the largest lodging value](charts/revenue_proxy.png)

## M08 Special requests and ADR
Mean ADR is 94.96 for no requests, 108.46 for one request and 117.03 for two requests. Pearson correlation is 0.172 and Spearman is 0.196, indicating a weak positive association. The five-request group has only 40 records. Guest composition, room type and season can explain part of the difference, so the result does not justify charging for requests.

Calculation: Group by request count; correlate total_of_special_requests with adr.

In [19]:
print(tables['requests'].to_string(index=False))
print(tables['correlations'].iloc[[1]].to_string(index=False))

 total_of_special_requests  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
                         0     70318     33556          0.477204    114.284920  94.961611     89.000    3.268651           0.0
                         1     33226      7318          0.220249     88.942425 108.455824    101.875    3.588455           1.0
                         2     12969      2866          0.220989     88.942170 117.033451    110.700    3.747089           2.0
                         3      2497       446          0.178614     92.569083 123.846131    117.850    3.977573           3.0
                         4       340        36          0.105882    110.520588 130.568706    119.425    4.317647           4.0
                         5        40         2          0.050000    105.550000 127.746500    110.440    4.650000           5.0
                  Feature Outcome      N  Pearson  Spearman
total_of_special_requests     adr 119390 0.172185  

## M09 Stay length for repeat and new guests
Repeat-guest bookings average 1.926 nights, compared with 3.477 for records without the repeat flag. There are 3,810 repeat-guest records and 115,580 others. The flag identifies recorded booking history, not a verified first-ever customer relationship. These means include canceled reservations.

Calculation: Add week and weekend nights; average by is_repeated_guest.

In [20]:
print(tables['guest_repeat'].to_string(index=False))

 is_repeated_guest  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
                 0    115580     43672          0.377851    106.425221 103.063492       95.0    3.477410      0.569484
                 1      3810       552          0.144882     30.786352  64.445997       65.0    1.925984      0.628346


## M10 Most frequently reserved room type
Room type A is most common, with 85,994 bookings, or 72.03% of the sample. Type D follows with 19,201. The analysis uses reserved_room_type as requested, rather than the final assigned room. Codes are anonymized and should not be renamed as standard, deluxe or suite without a mapping.

Calculation: Count records by reserved_room_type.

In [21]:
print(tables['rooms'].to_string(index=False))

reserved_room_type  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
                 A     85994     33630          0.391074    110.940240  90.796658     89.000    3.124613      0.493825
                 B      1118       368          0.329159    113.268336  90.360447     87.000    3.396243      0.775492
                 C       932       308          0.330472     80.173820 160.216792    162.995    4.669528      0.777897
                 D     19201      6102          0.317796     86.829488 120.682882    120.700    4.132858      0.779074
                 E      6535      1914          0.292884     91.121500 124.540179    114.750    4.719969      0.834277
                 F      2897       880          0.303763     68.529168 167.689651    177.000    3.718329      0.736969
                 G      2094       763          0.364374     80.248329 175.996027    175.350    4.295129      0.654728
                 H       601       245          

## A01 Factors associated with cancellation
A logistic regression on 119,206 eligible records finds higher cancellation odds with longer lead time, any previous cancellation, City Hotel, Online TA and non-refundable deposits, after the specified controls. Repeat-guest records have lower odds. An extra 30 lead-time days has an odds ratio of 1.095, with a 95% interval of 1.089 to 1.101. Odds ratios compare odds, not probability-point changes. The full coefficient table includes robust uncertainty and adjusted p-values.

Calculation: Fit an unpenalized logistic model with HC1 robust standard errors; apply Benjamini-Hochberg adjustment across non-intercept terms.

In [22]:
print(tables['cancellation_logistic'].to_string(index=False))
print(tables['cancellation_retrospective'].tail(2).to_string(index=False))
print(tables['prediction_metrics'].to_string(index=False))

                                       Term  Coefficient  RobustSE  OddsRatio     CI_Low    CI_High        PValue    FDR_PValue
                                  Intercept    -2.447280  0.043281   0.086529   0.079491   0.094189  0.000000e+00           NaN
                                  lead_time     0.090922  0.002709   1.095183   1.089383   1.101015 6.643198e-247 3.985919e-246
                                TotalNights     0.053230  0.003068   1.054672   1.048350   1.061032  1.893814e-67  6.493077e-67
                                     adults     0.132548  0.013438   1.141734   1.112055   1.172206  5.989050e-23  1.197810e-22
                                   children     0.179922  0.015945   1.197124   1.160289   1.235127  1.578376e-29  3.443729e-29
                                     babies    -0.451588  0.087468   0.636616   0.536320   0.755669  2.431222e-07  3.889955e-07
                          is_repeated_guest    -2.492122  0.110319   0.082734   0.066647   0.102705 5.42

## A02 Guest composition and ADR
In the adjusted multiple regression, an additional adult is associated with 5.93 more ADR units, a child with 14.67, and a baby with 5.90. The respective 95% intervals are 3.85 to 8.02, 13.75 to 15.60, and 3.27 to 8.54. The model holds hotel, segment, reserved room, arrival month, year and stay length constant. These are conditional associations, not per-person surcharges.

Calculation: OLS with adults, children and babies plus controls; HC3 standard errors; compare an upper-99th-percentile capped outcome.

In [23]:
print(tables['adr_regression'].to_string(index=False))
print(tables['adr_regression_capped'].head(5).to_string(index=False))

                                   Term  Coefficient  RobustSE      CI_Low     CI_High        PValue    FDR_PValue
                              Intercept    41.367755  1.699256   38.037213   44.698297 6.609391e-131           NaN
                                 adults     5.932366  1.064099    3.846731    8.018001  2.475139e-08  2.828730e-08
                               children    14.671378  0.472343   13.745585   15.597171 8.146035e-212 1.303366e-211
                                 babies     5.902042  1.343933    3.267933    8.536151  1.125097e-05  1.241487e-05
                            TotalNights    -0.200821  0.048232   -0.295355   -0.106287  3.131776e-05  3.340561e-05
      hotel: City Hotel vs Resort Hotel    18.906604  0.261007   18.395031   19.418177  0.000000e+00  0.000000e+00
     market_segment: Aviation vs Direct   -13.352650  1.636793  -16.560763  -10.144536  3.411422e-16  4.043167e-16
market_segment: Complementary vs Direct  -105.759111  1.364567 -108.433663 -103.

## A03 Booking changes and guest requests
Booking changes and special requests have Pearson correlation of 0.0528 and Spearman correlation of 0.0419. Both are close to zero. The data therefore offers little evidence of a strong overall relationship between these counts. More importantly, no satisfaction score is present. The requested relationship can be measured, but the effect of booking flexibility on guest satisfaction cannot.

Calculation: Correlate booking_changes and total_of_special_requests; inspect counts by change frequency.

In [24]:
print(tables['correlations'].iloc[[2]].to_string(index=False))
print(tables['changes'].to_string(index=False))

        Feature                   Outcome      N  Pearson  Spearman
booking_changes total_of_special_requests 119390 0.052833  0.041908
 booking_changes  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
               0    101314     41391          0.408542    104.531782 101.369789     94.990    3.357275      0.555609
               1     12701      1807          0.142272     98.151327 103.197860     91.500    3.641524      0.624911
               2      3805       766          0.201314    106.515637 107.285803     99.000    4.105125      0.712746
               3       927       144          0.155340    111.730313 109.797929     97.200    4.116505      0.810140
               4       376        67          0.178191    106.904255 107.641303     98.250    4.412234      0.805851
               5       118        20          0.169492    109.949153 103.151695     89.500    5.101695      0.822034
               6        63        18         

## A04 Seasonal cancellation patterns
Across all observed years, summer has the highest cancellation rate at 38.73% and winter the lowest at 33.08%. The complete year 2016 gives a different ranking: autumn is highest at 38.45%, followed by summer at 36.31%, spring at 34.67% and winter at 32.92%. This difference warns against assuming a stable seasonal rule from a partial multi-year sample. Seasonal terms are not significant at the adjusted 5% level in the baseline logistic model.

Calculation: Define Northern Hemisphere seasons; calculate canceled / bookings by season and month, with a separate complete-2016 view.

In [25]:
print(tables['season'].to_string(index=False))
print(tables['season_2016'].to_string(index=False))

Season  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
Autumn     28462     10484          0.368351    116.966236  90.868100       88.0    3.266777      0.517638
Spring     32674     12350          0.377976     95.654190  97.475909       93.6    3.350003      0.518302
Summer     37477     14516          0.387331    128.342850 128.768785      120.0    3.812952      0.664327
Winter     20777      6874          0.330847     55.519084  75.108733       74.8    3.076575      0.560716
Season  Bookings  Canceled  CancellationRate  MeanLeadDays    MeanADR  MedianADR  MeanNights  MeanRequests
Autumn     16051      6172          0.384524    129.540963  97.722435       95.0    3.264843      0.620584
Spring     15730      5453          0.346662     87.139606  88.503345       85.0    3.212142      0.422378
Summer     14927      5420          0.363100    121.489985 124.828132      116.1    3.748710      0.611375
Winter      9999      3292          0

![Seasonal cancellation patterns](charts/seasonal_cancellation.png)

## A05 Lead time across market segments
Groups book farthest ahead, with a median of 162 days and a middle-half range of 70 to 285 days. Offline TA/TO has a median of 109 days, Online TA 58, Direct 16 and Corporate 7. Aviation and Complementary each have a median of 3 days. Undefined has only two records. These distributions can help choose when to send confirmations and reminders for each segment.

Calculation: Group lead_time by market_segment; calculate median, quartiles and percentiles; draw a boxplot.

In [26]:
print(tables['lead_by_segment'].to_string(index=False))

market_segment  Bookings       Mean  Median    Q1     Q3   P90  Maximum
      Aviation       237   4.443038     3.0  1.00   6.00  11.0       23
 Complementary       743  13.286676     3.0  0.00  11.00  36.8      386
     Corporate      5295  22.125590     7.0  2.00  26.50  69.0      343
        Direct     12606  49.859115    16.0  1.00  75.00 158.0      737
        Groups     19811 186.973096   162.0 70.00 285.00 378.0      629
 Offline TA/TO     24219 135.004459   109.0 40.00 201.00 301.0      532
     Online TA     56477  82.998725    58.0 18.00 129.00 201.0      403
     Undefined         2   1.500000     1.5  1.25   1.75   1.9        2


![Lead time across market segments](charts/lead_segment_boxplot.png)

## Numerical verification
Compare the unpenalized logistic fit with an independent SciPy optimizer using the same design matrix. This checks numerical estimation, not the causal assumptions or independence of the records.

In [27]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
X = state['X']
y = state['model'].is_canceled.to_numpy(float)
beta = tables['cancellation_logistic'].Coefficient.to_numpy()
def objective(b):
    eta = X @ b
    return (np.logaddexp(0, eta).sum() - y @ eta) / len(y)
def gradient(b):
    return X.T @ (expit(X @ b) - y) / len(y)
check = minimize(objective, beta + 0.01, jac=gradient, method='BFGS', options={'gtol': 1e-9, 'maxiter': 1000})
assert np.max(np.abs(check.x - beta)) < 1e-4
print('Maximum coefficient difference:', np.max(np.abs(check.x - beta)))
print('Maximum mean score:', np.max(np.abs(gradient(beta))))

Maximum coefficient difference: 5.194462717073023e-06
Maximum mean score: 3.847809517018738e-15


## Scope and reuse
The source has no unique booking identifier, daily room inventory, actual revenue ledger or satisfaction score. Identical rows are retained in the primary results. The full duplicate sensitivity and flagged-record files are included. Regression uncertainty depends on observational assumptions. The workbook formulas update from its 52 monthly input rows; statistical tables are saved Python results.